### meteo.data.gouv.fr SIM2 Quotidien - Lecture des données du dernier mois (LATEST) & agrégation mensuelle
Ce script ne traite que le fichier LATEST Quotidien, contenant le mois précédent et le dernier mois en cours
- data: https://meteo.data.gouv.fr/datasets
- Auteur: L. Duffar

Ce script a un intérêt dans 2 cas :
- si le dernier mois est incomplet, cela permet d'avoir la valeur mensuelle partielle avant la diffusion par Météo-France 1 fois par mois de la valeur mensuelle officielle
- Certains paramètres n'existent pas sous forme mensuelle (comme Equivalent en eau du manteau neigeux), cela permettra à terme de constituer un historique depuis 1958



#### Téléchargement et lecture du fichier LATEST Quotidien

In [ ]:
############
# Auteur: L. Duffar
# Date: Juilet 2025
############
import pandas as pd
import plotly.express as px
import datetime
import requests
import os
import gzip

############### Personalisation
folder_csv_quot = r"X:\1-COMMUN\DIS\Documentation\Hydrologie\Documentation externe\Climat France\Météo-France\meteo.data\REFERENCE\SIM2\Q"
folder_csv_mens = r"X:\1-COMMUN\DIS\Documentation\Hydrologie\Documentation externe\Climat France\Météo-France\meteo.data\REFERENCE\SIM2\M"

file_csv_quot= "QUOT_SIM2_latest-20250601-20250719.csv" # généralement date de l'avant veille du jour du téléchargement (mais pas régulier)
file_csv_mens= "MENS_SIM2_current.csv"

download= True

############### Initialisation
print(datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))
url_gz= r"https://www.data.gouv.fr/fr/datasets/r/adcca99a-6db0-495a-869f-40c888174a57"

# Définition des Fonctions (téléchargement et de décompression ...)
def convert_to_date(date):
    return pd.to_datetime(str(date), format='%Y%m%d', errors='coerce')

def download_file(url, filename):
    filename= filename.split('.')[0] + '.gz'
    file= os.path.join(folder_gz, filename)
    print('Téléchargement: ', filename)
    response = requests.get(url)
    if response.status_code == 200:
        with open(file, 'wb') as f:
            f.write(response.content)
    else:
        print("Fichier d'archive non présent à l'url habituelle: ", file)

def decompress_gz(filename):
    filename= filename.split('.')[0] + '.gz'
    file= os.path.join(folder_gz, filename)
    
    if os.path.exists(file):
        with gzip.open(file, 'rb') as f_in:
            file= os.path.join(folder_csv_quot, filename.split('.')[0] + '.csv') 
            print('Décompression', filename)
            with open(file, 'wb') as f_out:
                f_out.write(f_in.read())
    else:
        print("Fichier d'archive non trouvé: ", file)

############### Décompression de l'archjive & Lecture du fichier CSV en pandas
if download:
    download_file(url_gz, file_csv_quot)
    decompress_gz(file_csv_quot)

df = pd.read_csv(folder_csv_quot + "\\" + file_csv_quot, skiprows= 0, header=0, sep=";")
# force numposte en string avec le zéro intial éventuel
df['DATE']= df['DATE'].apply(convert_to_date)

df

2025-07-Z20 11:49
Téléchargement:  QUOT_SIM2_latest-20250601-20250719.gz
Décompression QUOT_SIM2_latest-20250601-20250719.gz


,LAMBX,LAMBY,DATE,PRENEI_Q,PRELIQ_Q,T_Q,FF_Q,Q_Q,DLI_Q,SSI_Q,...,RESR_NEIGE6_Q,HTEURNEIGE_Q,HTEURNEIGE6_Q,HTEURNEIGEX_Q,SNOW_FRAC_Q,ECOULEMENT_Q,WG_RACINE_Q,WGI_RACINE_Q,TINF_H_Q,TSUP_H_Q
0,600,24010,2025-06-01,0.0,0.3,15.5,4.8,8.672,2962.4,971.8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.265,0.0,12.8,18.7
1,600,24010,2025-06-02,0.0,0.2,15.1,3.9,8.017,2983.0,1755.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.265,0.0,12.0,19.2
2,600,24010,2025-06-03,0.0,4.4,14.1,5.6,8.927,3138.6,528.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.263,0.0,10.7,15.6
3,600,24010,2025-06-04,0.0,1.6,14.4,5.3,8.602,2946.5,979.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.264,0.0,10.9,16.6
4,600,24010,2025-06-05,0.0,4.5,14.6,5.3,9.371,3308.5,541.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.264,0.0,14.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484703,11960,17450,2025-07-15,0.0,0.0,24.9,1.9,14.310,3371.8,2726.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.218,0.0,20.7,29.1
484704,11960,17450,2025-07-16,0.0,0.0,26.2,1.6,15.059,3428.4,2687.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.218,0.0,22.6,30.0
484705,11960,17450,2025-07-17,0.0,0.0,26.4,1.5,15.206,3427.3,2390.4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.217,0.0,22.2,31.2
484706,11960,17450,2025-07-18,0.0,0.0,26.4,2.2,14.846,3431.0,2584.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.217,0.0,20.6,31.0


#### Agrégation mensuelle du dernier mois

In [2]:
print(datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))
agreg_dict = {
    "LAMBX": ["first", "LAMBX"],
    "LAMBY": ["first", "LAMBY"],
    "DATE": ["date2datemois", "DATE"],
    "PRENEI_Q": ["sum", "PRENEI_MENS"],
    "PRELIQ_Q": ["sum", "PRELIQ_MENS"],
    "PE_Q": ["sum", "PE_MENS"],
    "T_Q": ["mean", "T_MENS"],
    "TINF_H_Q": ["mean", "TINF_H_Q_M"],
    "TSUP_H_Q": ["mean", "TSUP_H_Q_M"],
    "FF_Q": ["mean", "FF_Q_M"],
    "DLI_Q": ["sum", "DLI_Q_M"],
    "SSI_Q": ["sum", "SSI_Q_M"],
    "EVAP_Q": ["sum", "EVAP_MENS"],
    "ETP_Q": ["sum", "ETP_MENS"],
    "Q_Q": ["mean", "Q_Q_M"],
    "HU_Q": ["mean", "HU_Q_M"],
    "SWI_Q": ["-", "_"],
    "DRAINC_Q": ["sum", "DRAINC_MENS"],
    "RUNC_Q": ["sum", "RUNC_MENS"],
    "WG_RACINE_Q": ["end", "WG_RACINE_Q_M"],
    "WGI_RACINE_Q": ["end", "WGI_RACINE_Q_M"],
    "RESR_NEIGE_Q": ["mean", "RESR_NEIGE_Q_M"],
    "RESR_NEIGE6_Q": ["end", "RESR_NEIGE6_Q_M"],
    "HTEURNEIGE_Q": ["mean", "HTEURNEIGE_Q_M"],
    "HTEURNEIGE6_Q": ["end", "HTEURNEIGE6_Q_M"],
    "HTEURNEIGEX_Q": ["-", "-"],
    "SNOW_FRAC_Q": ["mean", "SNOW_FRAC_Q_M"],
    "ECOULEMENT_Q": ["sum", "ECOULEMENT_MENS"]
}

# effectuer une agrégation avec les règles suivantes:
# - le groupby s'effectue sur la combinaison de "LAMBX" "LAMBY" et mois de la colonne "DATE"
# - le nom de colonne agrégée est défini par le deuxième item de la liste agreg_list
# - l'opération d'agrégation est définie par le dictionnaire agreg_dict, dans le premier item de la liste
# - pour les variables de type "sum", on somme les valeurs
# - pour les variables de type "mean", on calcule la moyenne des valeurs
# - pour les variables de type "first", on prend la première valeur
# - pour les variables de type "end", on prend la dernière valeur

# définis la fonction d'agrégation selon les règles ci-dessus
def aggregate_data(df, agreg_dict):
    aggregation_functions = {}
    for key, value in agreg_dict.items():
        if key in ["LAMBX", "LAMBY"]:
            continue  # Skip LAMBX and LAMBY as they are used for grouping
        if value[0] == "sum":
            aggregation_functions[key] = "sum"
        elif value[0] == "mean":
            aggregation_functions[key] = "mean"
        elif value[0] == "first":
            aggregation_functions[key] = "first"
        elif value[0] == "end":
            aggregation_functions[key] = lambda x: x.iloc[-1]
        elif value[0] == "date2datemois":
            aggregation_functions[key] = lambda x: x.iloc[0].replace(day=1)

    # Add a column for the month
    df['MONTH'] = df['DATE'].dt.to_period('M')

    # Perform the groupby and aggregation
    df_agreg = df.groupby(["LAMBX", "LAMBY", "MONTH"]).agg(aggregation_functions).reset_index()

    # Rename columns according to the second item in the list, excluding LAMBX and LAMBY
    rename_dict = {key: value[1] for key, value in agreg_dict.items() if key not in ["LAMBX", "LAMBY"]}
    df_agreg.rename(columns=rename_dict, inplace=True)

    # Replace the 'MONTH' column with the first day of the month
    df_agreg['DATE'] = df_agreg['MONTH'].apply(lambda x: x.start_time)
    df_agreg.drop(columns=['MONTH'], inplace=True)

    return df_agreg

df_agreg = aggregate_data(df, agreg_dict)

df_agreg

2025-07-20 11:50


,LAMBX,LAMBY,DATE,PRENEI_MENS,PRELIQ_MENS,PE_MENS,T_MENS,TINF_H_Q_M,TSUP_H_Q_M,FF_Q_M,...,DRAINC_MENS,RUNC_MENS,WG_RACINE_Q_M,WGI_RACINE_Q_M,RESR_NEIGE_Q_M,RESR_NEIGE6_Q_M,HTEURNEIGE_Q_M,HTEURNEIGE6_Q_M,SNOW_FRAC_Q_M,ECOULEMENT_MENS
0,600,24010,2025-06-01,0.0,40.7,-28.0,17.353333,14.263333,20.660000,4.316667,...,13.3,1.7,0.249,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,600,24010,2025-07-01,0.0,26.0,-14.7,18.763158,15.642105,22.452632,4.657895,...,7.6,0.6,0.234,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,760,23610,2025-06-01,0.0,45.2,-26.0,17.863333,14.853333,21.200000,3.763333,...,12.4,1.5,0.186,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,760,23610,2025-07-01,0.0,18.9,-10.8,19.194737,16.052632,22.989474,4.121053,...,7.6,0.2,0.169,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,760,23930,2025-06-01,0.0,44.9,-23.5,17.973333,14.890000,21.350000,3.766667,...,12.5,2.3,0.240,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19779,11960,17210,2025-07-01,0.0,5.7,-13.8,26.131579,21.989474,30.552632,2.242105,...,3.9,0.0,0.198,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19780,11960,17290,2025-06-01,0.0,1.2,-52.2,24.896667,20.806667,29.656667,2.216667,...,6.4,0.0,0.229,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19781,11960,17290,2025-07-01,0.0,5.7,-20.3,26.131579,21.989474,30.552632,2.242105,...,3.8,0.0,0.216,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19782,11960,17450,2025-06-01,0.0,1.2,-47.2,25.010000,20.983333,29.656667,2.223333,...,7.0,0.0,0.226,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Créé un fichier CSV MENSUEL du dernier mois en cours qui pourra être utilisé en complément du fichier LATEST


In [3]:
print(datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))

# réduit le dataframe agrégé aux colonnes au dernier mois
df_agreg = df_agreg[df_agreg['DATE'] == df_agreg['DATE'].max()]

#  convert column 'DATE' to string YYYMMDD
df_agreg['DATE'] = df_agreg['DATE'].dt.strftime('%Y%m%d')

# enregistre le dataframe agrégé dans un fichier excel
df_agreg.to_csv(os.path.join(folder_csv_mens, file_csv_mens), index=False, sep=";")

print("Fichier csv CURRENT enregistré: ", file_csv_mens)

2025-07-20 11:50


C:\Users\DUFFAR\AppData\Local\Temp\ipykernel_20068\383879493.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_agreg['DATE'] = df_agreg['DATE'].dt.strftime('%Y%m%d')


Fichier csv CURRENT enregistré:  MENS_SIM2_current.csv
